# Gender Classification using Fingerprints

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
print(filename)  # As you can see, the filename contains useful information such as gender and finger type. 
# So we need to extract those words.

In [ ]:
dirname

In [ ]:
def extraction(img_path):  #We only need to extract gender
    filename = os.path.basename(img_path)
    gender = filename.split("__")[1].split("_")[0]
    return 0 if gender == 'M' else 1

In [ ]:
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
img_size=96

def load(path):
    print("Loading data from:", path)
    data = []

    for image in os.listdir(path):
        try:
            img_path = os.path.join(path, image)
            img_array = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img_resize = cv2.resize(img_array, (img_size, img_size))
            img_ready = img_resize.reshape(img_size, img_size, 1) / 255.0
            label = extraction(img_path)  
            data.append([label, img_ready])

        except Exception as e:
            print(f"Error: {e} -> {image}")
            continue

    return data

#We read the image, converted it to black and white, resized it, normalized it and added it to the list as a label-image.

In [ ]:
Real_path = "/kaggle/input/socofing/socofing/SOCOFing/Real"
Easy_path = "/kaggle/input/socofing/socofing/SOCOFing/Altered/Altered-Easy"
Medium_path = "/kaggle/input/socofing/socofing/SOCOFing/Altered/Altered-Medium"
Hard_path = "/kaggle/input/socofing/socofing/SOCOFing/Altered/Altered-Hard"


Easy_data = load(Easy_path)
Medium_data = load(Medium_path)
Hard_data = load(Hard_path)
test = load(Real_path)

data = Easy_data + Medium_data + Hard_data

del Easy_data, Medium_data, Hard_data


In [ ]:
labels, train_data = zip(*data) 
labels = np.array(labels)
train_data = np.stack(train_data)  

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model= Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(96, 96, 1)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(256, (3,3), activation="relu"),
    Conv2D(512, (3,3), activation="relu"),
    Flatten(),
    Dropout(0.2),
    Dense(250, activation="relu"),
    Dense(1, activation="sigmoid")
    ])

#We created the neural network model

In [ ]:
model.compile(optimizer="adam",loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
history=model.fit(train_data,labels,epochs=20, batch_size=32, verbose=1)

In [ ]:
model.summary()

In [ ]:
import matplotlib.pyplot as plt
pd.DataFrame(history.history).plot(figsize = (8,5))
plt.grid(True)
plt.gca().set_ylim(0,1)

In [ ]:
test_labels, test_data = zip(*test) 
test_labels = np.array(test_labels)
test_data = np.stack(test_data) 

In [ ]:
model.evaluate(test_data, test_labels)  #we are testing our model

### NOTE: This project has been developed as part of a course assignment. The original version was obtained from the following source: "https://amanxai.com/2020/08/02/gender-classification-model/". In accordance with the instructions, I studied the provided solution and then implemented my own approach to solve it independently.